# Systrophe tutorial

Hands-on walk through the package, from the single van Stockum cylinder to the off-set Tipler sinusoid pair, the time-machine harness, and the broader CTC zoo (Gödel, Gott, Kerr).

Run the cells in order. All numerical results match the whitepaper.

In [ ]:
import numpy as np
from systrophe import (
    VanStockumInterior, SystrophePair, SystropheArray,
    find_single_cylinder_windows, harness_time_loop,
    energy_condition_report,
    null_circular_omega, vanstockum_photon_omega,
    cauchy_horizon_estimate, tolman_blueshift_factor,
)
from systrophe.spacetimes import (
    GodelUniverse, GottPair, KerrSpacetime,
    LineSingularity, CosmicString,
)

## 1. Single van Stockum cylinder

Construct an `omega = R = 1` cylinder (so `a = 1`, supercritical), and inspect the closed-form analytic exterior.

In [ ]:
vs = VanStockumInterior(omega=1.0, R=1.0)
print(f"a = {vs.a}, alpha = {vs.alpha:.6f}")
print(f"exterior regime: {vs.regime}")
print(f"interior regime (a vs 1): {vs.interior_regime}")
for r in [1.5, 2.0, 3.0]:
    F = float(vs.analytic_exterior_F(r))
    K = float(vs.analytic_exterior_K(r))
    L = float(vs.analytic_exterior_L(r))
    print(f"r={r}: F={F:+.4f}, K={K:+.4f}, L={L:+.4f}, F*L+K^2={F*L+K**2:.4f} (should equal r^2={r**2})")

## 2. CTC bands and time-travel orbit

In [ ]:
windows = find_single_cylinder_windows(vs, r_min=1.001, r_max=200.0)
print(f"{len(windows)} CTC bands:")
for w in windows:
    print(f"  r in [{w.r_inner:.3f}, {w.r_outer:.3f}], deepest L = {w.L_min:.3f} at r = {w.r_min_L:.3f}")
result = harness_time_loop(windows[0], target_dt_per_rev=-1.0, n_revolutions=10)
print(f"\n10-revolution backward orbit:")
print(f"  Omega = {result['Omega']:.4f}, dt/rev = {result['dt_per_revolution']}, dtau/rev = {result['dtau_per_revolution']:.3f}")
print(f"  After 10 revs: total dt = {result['total_coord_time_advance']:.1f}, total dtau = {result['total_proper_time_advance']:.3f}")

## 3. Co-axial pair: tunable CTCs via phase offset

In [ ]:
for delta in [0.0, np.pi / 4, np.pi / 2, np.pi]:
    pair = SystrophePair.from_cylinders(vs, vs, delta_offset=delta)
    bands = pair.ctc_bands(r_min=1.05, r_max=20.0)
    print(f"delta = {delta:.4f}: {len(bands)} CTC bands, log measure = {sum(np.log(b/a) for a, b in bands):.4f}")

## 4. N-cylinder array: N-fold topological extinction

In [ ]:
for N in [3, 4, 5]:
    arr = SystropheArray.uniform_phase_comb(vs, N=N)
    bands = arr.ctc_bands(r_min=1.05, r_max=20.0)
    print(f"N = {N} uniform phase comb: {len(bands)} CTC bands (expected 0 -- phasor sum vanishes)")

## 5. Energy conditions: van Stockum dust is healthy matter

In [ ]:
report = energy_condition_report(vs)
print(f"NEC: {report.nec_holds}, WEC: {report.wec_holds}, SEC: {report.sec_holds}, DEC: {report.dec_holds}")
print(f"Total energy per unit z-length: {report.total_energy_per_unit_length:.4f} (in c=G=1 units)")

## 6. CTC zoo: Gödel, Gott, Kerr

In [ ]:
godel = GodelUniverse(a=1.0)
print(f"Gödel CTC threshold radius: {godel.ctc_threshold_radius:.4f}")
gott = GottPair(mu=0.05, v=0.7)
print(f"Gott pair (mu=0.05, v=0.7): has CTC = {gott.has_ctc()}")
kerr = KerrSpacetime(M=1.0, a=0.9)
print(f"Kerr (M=1, a=0.9) equatorial CTC radius: {kerr.equatorial_ctc_radius():.4f}")

## 7. Singularity reinterpretations

In [ ]:
# (1) Rotating line singularity (Lewis/Tipler)
ls = LineSingularity(omega=1.0, R_match=1.0)
print(f"Line singularity: regime = {ls.regime}")
print(f"  M per length = {ls.equivalent_van_stockum().mass_per_unit_length:.4f}")

# (2) Cosmic string (Vilenkin) -> Gott pair
cs = CosmicString(mu=0.05)
print(f"\nCosmic string mu=0.05: deficit = {cs.deficit_angle:.4f} rad")
pair = cs.compose_with_gott(cs, v=0.8)
print(f"  composed with v=0.8: has CTC = {pair.has_ctc()}")

# (3) Kerr ring singularity
k = KerrSpacetime(M=1.0, a=0.9)
print(f"\nKerr (M=1, a=0.9): horizons = {k.event_horizons}")

## 8. Photon orbits and chronology-protection diagnostic

In [ ]:
rs = np.array([1.5, 2.0, 3.0])
om_minus, om_plus = vanstockum_photon_omega(vs, rs)
print(f"Photon Omega_+- at r = 1.5, 2, 3:")
for r, om_m, om_p in zip(rs, om_minus, om_plus):
    print(f"  r = {r}: Omega_- = {om_m:+.4f}, Omega_+ = {om_p:+.4f}")

horizons = cauchy_horizon_estimate(vs)
print(f"\nCauchy horizon candidate radii (F = 0): {horizons[:3]}")